# **Section 1: Install Dependencies**

In [56]:
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers
!pip install -q pypdf

# **Section 2: Upload PDF**

In [61]:
from google.colab import files

uploaded = files.upload()

Saving ADITIPAITANDY_2411022250019 (3).pdf to ADITIPAITANDY_2411022250019 (3) (2).pdf


In [62]:
from langchain_community.document_loaders import PyPDFLoader

pdf_name = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_name)
docs = loader.load()

print("Total Pages:", len(docs))

Total Pages: 78


# **Section 3: Chunk Documents**

In [63]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

print("Chunks:", len(chunks))

Chunks: 173


# **Section 4: Create Embeddings + FAISS**

In [64]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(
    chunks,
    embedding_model
)

print("Vector DB created!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector DB created!


# **Section 5: Load LLM**

In [65]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

print("LLM Loaded")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LLM Loaded


# **Section 6: Ask Questions**

In [66]:
def ask_rag(question):
    docs = db.similarity_search(question, k=3)

    context = "\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
Use the context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm(
        prompt,
        max_new_tokens=80,
        do_sample=False
    )

    return response[0]["generated_text"]

In [67]:
print(
    ask_rag(
        "What technologies were used in this project?"
    )
)

Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the context to answer the question.

Context:
55  
 
                              FIG: Crack Image                                                 FIG: No Crack Image 
 
A.2 Technologies Used: 
The following technologies and tools were used during project development: 
• Python  
• TensorFlow  
• Keras  
• OpenCV  
• NumPy  
• Pandas  
• Plotly  
• Streamlit  
• Google Colab  
• Visual Studio Code
33  
Hardware Configuration 
 
A setup built for testing carried these components. Running tasks required specific tools on hand. 
Tools present matched what the work needed. Equipment listed supported every task tested 
 
• Intel Core i5 or i7 processor 
• NVIDIA GPU support (for accelerated training) 
• 16GB/32GB RAM 
• Windows operating system 
 
With GPU acceleration available, tackling heavy computations - like training models or proces sing 
images - became faster. Efficiency improved noticeably when hardware support entered the 
workflow. Tasks once slow now moved at greater spee